# Módulo 7 · Clase 2 — Preparación de datos para ML

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | La regla de oro: fit en train, transform en todo |
| 2 | Codificación de variables categóricas |
| 3 | Escalado de variables numéricas |
| 4 | Imputación de valores missing |
| 5 | ColumnTransformer: procesar columnas distintas juntas |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    LabelEncoder, OrdinalEncoder, OneHotEncoder,
    StandardScaler, MinMaxScaler, RobustScaler
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

np.random.seed(42)
print("Librerías listas")

---
## 1. La regla de oro: fit solo en train

```
CONTAMINACIÓN DE DATOS (data leakage):
  El modelo aprende información del futuro (test) durante el entrenamiento.
  → El modelo parece mejor de lo que realmente es en producción.

MAL (leakage):
  scaler.fit(X)              # usa TODO el dataset → contamina
  X_scaled = scaler.transform(X)
  X_tr, X_te = train_test_split(X_scaled)

CORRECTO:
  X_tr, X_te = train_test_split(X)  # dividir PRIMERO
  scaler.fit(X_tr)                   # aprende estadísticos SOLO de train
  X_tr_sc = scaler.transform(X_tr)   # aplica a train
  X_te_sc = scaler.transform(X_te)   # aplica a test (SIN refitear)

Mismo principio para: scalers, encoders, imputers, PCA...
```

In [ ]:
# Demostrar el efecto del leakage
np.random.seed(42)
X_demo = np.random.randn(100, 1) * 10 + 50  # media=50, std=10
# Añadimos un outlier extremo solo en test
X_demo[90] = 200

X_tr, X_te = train_test_split(X_demo, test_size=0.2, random_state=42)

# MAL: fit en todo → el outlier de test contamina el scaler
scaler_mal = StandardScaler()
scaler_mal.fit(X_demo)
print(f"Scaler MAL (fit en todo): media={scaler_mal.mean_[0]:.2f}, std={scaler_mal.scale_[0]:.2f}")

# BIEN: fit solo en train
scaler_bien = StandardScaler()
scaler_bien.fit(X_tr)
print(f"Scaler BIEN (fit en train): media={scaler_bien.mean_[0]:.2f}, std={scaler_bien.scale_[0]:.2f}")

print("\nEl scaler 'malo' tiene sus parámetros afectados por el outlier de test.")
print("En producción nunca tendremos acceso a datos futuros → el scaler BIEN es el correcto.")

---
## 2. Codificación de variables categóricas

```
Los algoritmos de ML trabajan con números → debemos convertir texto.

LABEL ENCODER:    A→0, B→1, C→2  (implica orden: C > B > A)
  ✓ Usar solo para la variable TARGET (y)
  ✗ No usar para features → introduce orden artificial

ORDINAL ENCODER:  ['bajo','medio','alto'] → [0, 1, 2]
  ✓ Usar cuando SÍ hay orden real: talla, nivel educativo

ONE-HOT ENCODER:  'rojo' → [1,0,0], 'verde' → [0,1,0]
  ✓ Usar cuando NO hay orden: ciudad, color, marca
  ⚠ Con muchas categorías → muchas columnas (considera target encoding)
```

In [ ]:
# Dataset de ejemplo
datos = pd.DataFrame({
    'nivel':    ['bajo', 'medio', 'alto', 'medio', 'bajo', 'alto'],
    'ciudad':   ['Madrid', 'Barcelona', 'Madrid', 'Valencia', 'Sevilla', 'Barcelona'],
    'compra':   ['si', 'no', 'si', 'si', 'no', 'si'],
})
print("Datos originales:")
print(datos)

# OrdinalEncoder para 'nivel' (tiene orden real)
ord_enc = OrdinalEncoder(categories=[['bajo', 'medio', 'alto']])
datos['nivel_cod'] = ord_enc.fit_transform(datos[['nivel']]).astype(int)

# OneHotEncoder para 'ciudad' (sin orden)
ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' evita multicolinealidad
ciudad_ohe = ohe.fit_transform(datos[['ciudad']])
ciudad_df  = pd.DataFrame(ciudad_ohe, columns=ohe.get_feature_names_out(['ciudad']))

# LabelEncoder para target 'compra'
le = LabelEncoder()
datos['compra_num'] = le.fit_transform(datos['compra'])
print(f"LabelEncoder clases: {dict(zip(le.classes_, le.transform(le.classes_)))}")

resultado = pd.concat([datos, ciudad_df], axis=1)
print("\nDatos codificados:")
print(resultado)

# Problema del orden artificial con LabelEncoder en features
print("\n=== ¿Por qué NO usar LabelEncoder en features? ===")
ciudades = ['Madrid', 'Barcelona', 'Valencia', 'Sevilla']
le_ciudad = LabelEncoder()
cod = le_ciudad.fit_transform(ciudades)
print(f"LabelEncoder: {dict(zip(ciudades, cod))}")
print("→ Sevilla=2, Valencia=3 → ¿Valencia es 50% más que Sevilla? ¡No tiene sentido!")

---
## 3. Escalado de variables numéricas

```
¿Por qué escalar?
  Sin escalado: 'metros²' (50-200) domina sobre 'nº habitaciones' (1-5)
  → Los gradientes en regresión logística/SVM/KNN se distorsionan

¿Cuándo NO hace falta?
  Decision Tree y Random Forest → no les afecta el escalado

STANDARD SCALER (Z-score):  x' = (x - μ) / σ  → media=0, std=1
  ✓ Uso general, especialmente con distribución normal
  ✗ Sensible a outliers

MIN-MAX SCALER:  x' = (x - min) / (max - min)  → [0, 1]
  ✓ Cuando necesitas valores en rango fijo (redes neuronales)
  ✗ Muy sensible a outliers

ROBUST SCALER:  x' = (x - mediana) / IQR  → robusto a outliers
  ✓ Cuando hay outliers que no puedes eliminar
```

In [ ]:
# Dataset con outliers
np.random.seed(42)
datos_num = np.array([10, 12, 11, 13, 10, 12, 100, 11, 13, 10], dtype=float)  # 100 es outlier

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler':   MinMaxScaler(),
    'RobustScaler':   RobustScaler(),
}

print(f"Datos originales: {datos_num}")
print(f"{'Scaler':20} {'Valor 10':>12} {'Valor 12':>12} {'Outlier 100':>14}")
for nombre, sc in scalers.items():
    transformados = sc.fit_transform(datos_num.reshape(-1,1)).ravel()
    print(f"{nombre:20} {transformados[0]:>12.4f} {transformados[1]:>12.4f} {transformados[6]:>14.4f}")

# Visualización
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
titulos = ['Original'] + list(scalers.keys())
datasets_viz = [datos_num] + [
    sc.fit_transform(datos_num.reshape(-1,1)).ravel()
    for sc in scalers.values()
]

for ax, titulo, d in zip(axes, titulos, datasets_viz):
    ax.boxplot(d, vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               flierprops=dict(marker='o', color='red', ms=8))
    ax.set_title(titulo, fontsize=10)
    ax.set_xticklabels([])

plt.suptitle('Efecto del escalado (punto rojo = outlier)', fontsize=11)
plt.tight_layout(); plt.show()

print("\nConclusión:")
print("  StandardScaler: el outlier queda en 7σ — muy visible pero altera los demás")
print("  MinMaxScaler: todos los valores normales se comprimen entre 0 y 0.03")
print("  RobustScaler: los valores normales mantienen escala, outlier queda en ~9")

---
## 4. Imputación de valores missing

In [ ]:
# Dataset con missings
X_con_nan = np.array([
    [1.0, 2.0, np.nan],
    [3.0, np.nan, 4.0],
    [np.nan, 6.0, 7.0],
    [8.0, 8.0, 9.0],
    [5.0, 4.0, 3.0],
])

estrategias = ['mean', 'median', 'most_frequent', 'constant']
print("Datos originales:")
print(X_con_nan)
print()

for estrategia in estrategias:
    kwargs = {'fill_value': -999} if estrategia == 'constant' else {}
    imputer = SimpleImputer(strategy=estrategia, **kwargs)
    X_imp   = imputer.fit_transform(X_con_nan)
    print(f"Estrategia '{estrategia}':")
    print(X_imp.round(2))
    print()

print("Regla de selección de estrategia:")
print("  mean:          datos numéricos sin outliers")
print("  median:        datos numéricos con outliers")
print("  most_frequent: datos categóricos")
print("  constant:      cuando el missing tiene significado propio (ej: sin respuesta)")

---
## 5. ColumnTransformer: todo junto

In [ ]:
# Dataset realista: predecir salario
np.random.seed(42)
n = 200
df_salario = pd.DataFrame({
    'experiencia': np.random.randint(0, 20, n).astype(float),
    'edad':        np.random.randint(22, 55, n).astype(float),
    'nivel_edu':   np.random.choice(['bachiller','grado','master','doctorado'], n),
    'ciudad':      np.random.choice(['Madrid','Barcelona','Valencia','Sevilla'], n),
    'salario':     None  # target
})
# Añadir missings
df_salario.loc[np.random.choice(n, 20, replace=False), 'experiencia'] = np.nan
df_salario.loc[np.random.choice(n, 10, replace=False), 'edad']        = np.nan

# Target: función del nivel educativo y experiencia
nivel_map = {'bachiller': 1, 'grado': 2, 'master': 3, 'doctorado': 4}
nivel_num = df_salario['nivel_edu'].map(nivel_map)
exp_clean = df_salario['experiencia'].fillna(df_salario['experiencia'].median())
df_salario['salario'] = (25000 + nivel_num * 8000 + exp_clean * 1500
                         + np.random.normal(0, 3000, n))

X = df_salario.drop('salario', axis=1)
y = df_salario['salario']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Definir qué hacer con cada tipo de columna
cols_numericas    = ['experiencia', 'edad']
cols_ordinales    = ['nivel_edu']
cols_nominales    = ['ciudad']

preprocesador = ColumnTransformer(transformers=[
    ('num', 
     # Pipeline de pasos para numéricas
     # 1. Imputar missing, 2. Escalar
     # (usaremos Pipeline en C9; aquí lo hacemos en dos pasos)
     StandardScaler(),
     cols_numericas),
    ('ord',
     OrdinalEncoder(categories=[['bachiller','grado','master','doctorado']]),
     cols_ordinales),
    ('nom',
     OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'),
     cols_nominales),
], remainder='drop')  # 'passthrough' para mantener el resto sin cambio

# Fit solo en train, transform en ambos
X_train_t = preprocesador.fit_transform(X_train)
X_test_t  = preprocesador.transform(X_test)

print(f"Forma original X_train:      {X_train.shape}")
print(f"Forma transformada X_train:  {X_train_t.shape}")
print(f"  Numéricas: {len(cols_numericas)} → {len(cols_numericas)} (escaladas)")
print(f"  Ordinales: {len(cols_ordinales)} → {len(cols_ordinales)} (entero)")
print(f"  Nominales: {len(cols_nominales)} → {X_test_t.shape[1]-len(cols_numericas)-len(cols_ordinales)} (OHE con drop)")

print("\nPrimeras 3 filas transformadas:")
print(X_train_t[:3].round(3))

# Entrenar con datos preprocesados
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
modelo = LinearRegression().fit(X_train_t, y_train)
r2 = r2_score(y_test, modelo.predict(X_test_t))
print(f"\nR² del modelo con preprocesamiento correcto: {r2:.4f}")

---
## Práctica

### Ejercicio 1 — Codificar el dataset Titanic

**Enunciado:** Carga el Titanic con seaborn y prepara las features:

```python
import seaborn as sns
df = sns.load_dataset('titanic')[['survived','pclass','sex','age','fare','embarked','alone']].copy()
```

1. Imputa `age` con la mediana y `embarked` con la moda
2. Codifica `sex` con LabelEncoder y `embarked` con OneHotEncoder
3. Escala `age` y `fare` con StandardScaler
4. Divide en train/test y verifica que el scaler se ajusta solo en train
5. Muestra la forma final de X_train y X_test

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

df = sns.load_dataset('titanic')[['survived','pclass','sex','age','fare','embarked','alone']].copy()

# 1. Imputar
df['age']      = SimpleImputer(strategy='median').fit_transform(df[['age']]).ravel()
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# 2. Codificar sex
le = LabelEncoder()
df['sex_num'] = le.fit_transform(df['sex'])  # female=0, male=1

ohe = OneHotEncoder(sparse_output=False, drop='first')
emb_arr = ohe.fit_transform(df[['embarked']])
emb_df  = pd.DataFrame(emb_arr, columns=ohe.get_feature_names_out(['embarked']), index=df.index)

df_final = pd.concat([df[['survived','pclass','sex_num','age','fare','alone']], emb_df], axis=1)
df_final['alone'] = df_final['alone'].astype(int)

X = df_final.drop('survived', axis=1)
y = df_final['survived']

# 3 y 4. Split → fit scaler solo en train
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cols_escalar = ['age', 'fare']
scaler = StandardScaler()
X_tr[cols_escalar] = scaler.fit_transform(X_tr[cols_escalar])  # fit EN TRAIN
X_te[cols_escalar] = scaler.transform(X_te[cols_escalar])      # solo transform en test

# 5. Forma final
print(f"X_train: {X_tr.shape}  |  X_test: {X_te.shape}")
print(f"Columnas: {list(X_tr.columns)}")
print(X_tr.head())
```
</details>

---
### Ejercicio 2 — Comparar escaladores en un modelo

**Enunciado:** Sobre el dataset de diabetes (`datasets.load_diabetes()`):

1. Entrena tres modelos con los mismos datos pero diferente escalado: sin escalar, StandardScaler, MinMaxScaler
2. Compara R² en test
3. ¿Por qué para LinearRegression el escalado no cambia el R²? ¿Y para qué modelos sí importaría?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import r2_score
import numpy as np

X, y = datasets.load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

resultados = {}

# Sin escalar
m = LinearRegression().fit(X_tr, y_tr)
resultados['Sin escalar'] = r2_score(y_te, m.predict(X_te))

# StandardScaler
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)
m2 = LinearRegression().fit(X_tr_s, y_tr)
resultados['StandardScaler'] = r2_score(y_te, m2.predict(X_te_s))

# MinMaxScaler
mm = MinMaxScaler()
X_tr_m = mm.fit_transform(X_tr)
X_te_m = mm.transform(X_te)
m3 = LinearRegression().fit(X_tr_m, y_tr)
resultados['MinMaxScaler'] = r2_score(y_te, m3.predict(X_te_m))

for nombre, r2 in resultados.items():
    print(f"{nombre:20}: R² = {r2:.6f}")

print("\nExplicación:")
print("  LinearRegression: el R² es IGUAL con o sin escalado.")
print("  Razón: escalar es una transformación lineal, los hiperplanos son equivalentes.")
print("  Los coeficientes cambian, pero las predicciones finales son idénticas.")
print()
print("  Modelos donde SÍ importa escalar:")
print("  - KNN: usa distancias, una feature grande domina")
print("  - SVM: el kernel RBF depende de distancias")
print("  - LogisticRegression con regularización")
print("  - Redes neuronales")
print("  - K-Means: usa distancias euclídeas")
```
</details>

---
### Ejercicio 3 — ColumnTransformer completo

**Enunciado:** Diseña un ColumnTransformer para este dataset de préstamos:

```python
np.random.seed(10)
n = 300
df_prestamo = pd.DataFrame({
    'ingreso':    np.random.normal(35000, 12000, n).clip(10000),
    'edad':       np.random.randint(18, 70, n).astype(float),
    'deuda':      np.random.exponential(5000, n),
    'empleo':     np.random.choice(['empleado','autonomo','desempleado'], n),
    'educacion':  np.random.choice(['primaria','secundaria','universitaria'], n),
    'aprobado':   np.random.randint(0, 2, n),
})
# Añadir 15 missings en 'ingreso' y 'edad'
df_prestamo.loc[np.random.choice(n,15), 'ingreso'] = np.nan
df_prestamo.loc[np.random.choice(n,10), 'edad']    = np.nan
```

1. Numéricas (ingreso, edad, deuda): imputa con mediana + StandardScaler
2. Nominal (empleo): OneHotEncoder
3. Ordinal (educacion): OrdinalEncoder con orden correcto
4. Divide en train/test y aplica el ColumnTransformer
5. Entrena un LinearRegression y evalúa

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

np.random.seed(10)
n = 300
df = pd.DataFrame({
    'ingreso':   np.random.normal(35000, 12000, n).clip(10000),
    'edad':      np.random.randint(18, 70, n).astype(float),
    'deuda':     np.random.exponential(5000, n),
    'empleo':    np.random.choice(['empleado','autonomo','desempleado'], n),
    'educacion': np.random.choice(['primaria','secundaria','universitaria'], n),
    'aprobado':  np.random.randint(0, 2, n),
})
df.loc[np.random.choice(n,15), 'ingreso'] = np.nan
df.loc[np.random.choice(n,10), 'edad']    = np.nan

X = df.drop('aprobado', axis=1)
y = df['aprobado']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 1. Numéricas: impute + scale
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

ct = ColumnTransformer([
    ('num', num_transformer, ['ingreso','edad','deuda']),
    ('nom', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), ['empleo']),
    ('ord', OrdinalEncoder(categories=[['primaria','secundaria','universitaria']]), ['educacion']),
])

X_tr_t = ct.fit_transform(X_tr)
X_te_t = ct.transform(X_te)

print(f"X_train transformada: {X_tr_t.shape}")
print(f"X_test transformada:  {X_te_t.shape}")

modelo = LogisticRegression(max_iter=1000).fit(X_tr_t, y_tr)
print(f"\nAccuracy en test: {modelo.score(X_te_t, y_te):.4f}")
```
</details>

---
### Ejercicio 4 — Detectar y corregir leakage

**Enunciado:** El siguiente código tiene un error de data leakage. Identifícalo, explica por qué es un problema y corrígelo:

```python
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes

X, y = load_diabetes(return_X_y=True)

# ¿Dónde está el leakage?
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # ← PROBLEMA

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)
modelo = LinearRegression().fit(X_train, y_train)
print(modelo.score(X_test, y_test))
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ: explica el problema y escribe la versión correcta


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X, y = load_diabetes(return_X_y=True)

print("PROBLEMA en el código original:")
print("  scaler.fit_transform(X) usa TODOS los datos (train + test).")
print("  → La media y std del scaler están contaminadas por los datos de test.")
print("  → El modelo 've' información de test antes de tiempo.")
print("  En producción los datos nuevos no existen durante el entrenamiento,")
print("  por lo que las métricas en test son demasiado optimistas.")

# VERSIÓN CON LEAKAGE (MAL)
scaler_mal = StandardScaler()
X_scaled   = scaler_mal.fit_transform(X)
X_tr_mal, X_te_mal, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
r2_mal = LinearRegression().fit(X_tr_mal, y_tr).score(X_te_mal, y_te)

# VERSIÓN CORRECTA
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_bien = StandardScaler()
X_tr_sc = scaler_bien.fit_transform(X_tr)  # fit SOLO en train
X_te_sc = scaler_bien.transform(X_te)      # transform en test
r2_bien = LinearRegression().fit(X_tr_sc, y_tr).score(X_te_sc, y_te)

print(f"\nR² CON leakage:    {r2_mal:.6f}")
print(f"R² SIN leakage:    {r2_bien:.6f}")
print("\nEn este caso la diferencia es pequeña (LinearRegression es estable),")
print("pero con scalers muy dependientes de outliers la diferencia puede ser significativa.")
```
</details>

---
### Ejercicio 5 (reto) — Ingeniería de features

**Enunciado:** La ingeniería de features es transformar los datos para que el modelo aprenda mejor. Sobre el dataset de casas:

```python
np.random.seed(42)
n = 500
df_casas = pd.DataFrame({
    'metros':      np.random.uniform(40, 250, n),
    'habitaciones':np.random.randint(1, 7, n),
    'antiguedad':  np.random.randint(0, 50, n),
    'planta':      np.random.randint(0, 12, n),
    'zona':        np.random.choice(['A','B','C','D'], n),
})
df_casas['precio'] = (1800*df_casas['metros'] + 15000*df_casas['habitaciones']
                      - 500*df_casas['antiguedad'] + 2000*df_casas['planta']
                      + df_casas['zona'].map({'A':50000,'B':30000,'C':10000,'D':0})
                      + np.random.normal(0, 10000, n))
```

1. Crea features derivadas: `metros_por_hab = metros/habitaciones`, `es_nuevo = antiguedad < 5`, `zona_premium = zona in ['A','B']`
2. Preprocesa correctamente (ColumnTransformer)
3. Compara R² con features originales vs features enriquecidas
4. ¿Cuánto mejora el modelo con ingeniería de features?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

np.random.seed(42)
n = 500
df = pd.DataFrame({
    'metros':       np.random.uniform(40, 250, n),
    'habitaciones': np.random.randint(1, 7, n),
    'antiguedad':   np.random.randint(0, 50, n),
    'planta':       np.random.randint(0, 12, n),
    'zona':         np.random.choice(['A','B','C','D'], n),
})
df['precio'] = (1800*df['metros'] + 15000*df['habitaciones']
                - 500*df['antiguedad'] + 2000*df['planta']
                + df['zona'].map({'A':50000,'B':30000,'C':10000,'D':0})
                + np.random.normal(0, 10000, n))

y = df['precio']

def preprocesar_y_evaluar(X, nombre):
    cols_num = X.select_dtypes(include='number').columns.tolist()
    cols_cat = X.select_dtypes(include='object').columns.tolist()
    ct = ColumnTransformer([
        ('num', StandardScaler(), cols_num),
        ('cat', OneHotEncoder(sparse_output=False, drop='first'), cols_cat),
    ]) if cols_cat else ColumnTransformer([('num', StandardScaler(), cols_num)])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    X_tr_t = ct.fit_transform(X_tr)
    X_te_t = ct.transform(X_te)
    modelo = LinearRegression().fit(X_tr_t, y_tr)
    r2 = r2_score(y_te, modelo.predict(X_te_t))
    print(f"{nombre:35}: R² = {r2:.4f}")
    return r2

# Versión original
X_orig = df.drop('precio', axis=1)
r2_orig = preprocesar_y_evaluar(X_orig, 'Features originales')

# Versión enriquecida
df_eng = df.copy()
df_eng['metros_por_hab'] = df_eng['metros'] / df_eng['habitaciones']
df_eng['es_nuevo']       = (df_eng['antiguedad'] < 5).astype(int)
df_eng['zona_premium']   = df_eng['zona'].isin(['A','B']).astype(int)
X_eng = df_eng.drop('precio', axis=1)
r2_eng = preprocesar_y_evaluar(X_eng, 'Features enriquecidas')

print(f"\nMejora con ingeniería de features: +{(r2_eng - r2_orig)*100:.2f} puntos de R²")
```
</details>

---
## Resumen

```python
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# REGLA DE ORO: fit solo en train
X_tr, X_te = train_test_split(X)
scaler.fit(X_tr)              # aprende estadísticos de train
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)  # NO fit de nuevo

# ColumnTransformer: distintas transformaciones por columna
ct = ColumnTransformer([
    ('num', StandardScaler(),              ['col1', 'col2']),
    ('cat', OneHotEncoder(drop='first'),   ['col3']),
    ('ord', OrdinalEncoder(categories=[[...]]), ['col4']),
])
X_tr_t = ct.fit_transform(X_tr)
X_te_t = ct.transform(X_te)
```

## ¿Qué viene en la próxima clase?
En la **Clase 3** aplicamos todo lo aprendido para entrenar modelos de regresión y evaluarlos con métricas como MSE, MAE y R².